# GenAI & Metadata-Driven Engineering

**Autor:** Maximilian Pazer | **Matrikelnummer:** 9545148  
**Kurs:** WWI2023F Data Management Fundamentals | **Datum:** 15.02.2026  
**Prüfer:** Andreas Buckenhofer

---

## Inhaltsverzeichnis

1. Einleitung
2. Star Schema Modelling
3. Large Language Models für strukturierte Datengenerierung
4. Methodik
5. Implementierung
6. Evaluation & Ergebnisse
7. Kritische Würdigung
8. Fazit
9. Quellen
10. GenAI Erklärung

---


In [1]:
# ===== Load project files from GitHub =====
!git clone https://github.com/maximpazer/starschema-llm-demo.git
%cd starschema-llm-


Cloning into 'starschema-llm-demo'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 156.86 KiB | 4.75 MiB/s, done.
[Errno 2] No such file or directory: 'starschema-llm-'
/content


In [2]:
!pip install openai duckdb --quiet

import duckdb
import os
import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
client = OpenAI()


OpenAI API key: ··········


# 1. Einleitung



Data Warehouses sind zentral für die analytische Auswertung von Unternehmensdaten. Dimensional Modeling nach Kimball gilt als de-facto-Standard für die relationale Data-Warehouse-Modellierung [1]. Der Prozess erfordert manuelle Designentscheidungen: Data Engineers wählen Geschäftsprozesse aus, definieren das Grain der Faktentabelle, schneiden Dimensionen zu und legen die Typen der Slowly Changing Dimensions (SCD) fest. Bei Dutzenden Data Marts ist dieser Prozess zeitaufwendig, fehleranfällig und kaum skalierbar.

Large Language Models (LLMs) zeigen beachtliche Erfolge bei der Generierung von strukturiertem Code aus natürlicher Sprache und könnten das Star-Schema-Design automatisieren [2]. Gleichzeitig dokumentieren Studien zu Text-to-SQL-Systemen Halluzinationen: LLMs erfinden Tabellen oder Spalten (schema-based hallucinations) oder erzeugen fehlerhafte logische Strukturen [11]. Die zentrale Forschungsfrage lautet daher, ob LLMs semantische Modellierungsentscheidungen (SCD-Typ-Wahl, Grain-Definition) aus User Stories korrekt ableiten können.

Ziel dieser Arbeit ist es zu untersuchen, ob ein LLM aus User Stories und Staging-Schemas automatisch ein korrektes Star Schema (DDL) generieren kann. Als repräsentatives Beispiel dient ein Automotive-Sales-Analytics-Szenario mit vier Dimensionen (Fahrzeug, Händler, Kunde, Zeit) und einer Faktentabelle (Verkäufe). Die Evaluation erfolgt durch den Vergleich mit einem manuell erstellten Golden Standard und analysiert, inwiefern Halluzinationen ein zentrales Problem der automatischen Modellierung darstellen.

Die Untersuchung beschränkt sich auf die konzeptionelle Schema-Modellierung (DDL-Generierung) eines einzelnen Star Schemas. Nicht betrachtet werden die physische Datenbefüllung (ETL-Prozesse), Query-Generierung, Performanceoptimierung oder Snowflake-Schemata. Die Ergebnisse sind daher auf die automatisierte Ableitung dimensionaler Modellierungsentscheidungen aus Metadaten und Anforderungen begrenzt.


---

# 2. Star Schema Modellierung


## 2.1 Dimensionen und Fakten

Das Star Schema ist ein dimensionales Datenmodell für Data Warehouses, das performante Endbenutzerabfragen ermöglicht und die Analyse von Metriken unterstützt [3, Folie 43].
Es besteht aus einer zentralen Faktentabelle, die von mehreren Dimensionstabellen umgeben ist [1; 3, Folie 45]. Abbildung 1 zeigt die sternförmige Anordnung, die dem Modell seinen Namen gibt.

![Star Schema](https://github.com/maximpazer/starschema-llm-demo/blob/main/images/Star_Schema.png?raw=1)

*Abbildung 1: Star Schema Modell (Quelle: [3, Folie 48])*

Die zentrale **Faktentabelle** enthält numerische Kennzahlen wie Umsatz, Verkaufsmenge oder Rabatt [3, Folie 45]. **Dimensionstabellen** beschreiben hingegen die analysierten Entitäten durch Attribute, die für Auswertungen benötigt werden. Dimensionen liefern damit den Kontext zu den Fakten, beispielsweise Kunde, Produkt oder Zeit [3, Folie 45]. Jede Dimension besitzt einen Primärschlüssel, der als Fremdschlüssel in der Faktentabelle referenziert wird. Kimball empfiehlt Surrogate Keys als künstlich generierte Primärschlüssel für die Verknüpfung zwischen Fakt- und Dimensionstabellen, während Natural Keys als fachliche Geschäftsschlüssel erhalten bleiben [3, Folie 18]. Durch Denormalisierung ermöglicht das Star Schema schnelle Abfragen ohne komplexe Join-Operationen [3, Folie 46].

## 2.2 Slowly Changing Dimensions (SCD)

Slowly Changing Dimensions (SCDs) beschreiben Strategien zur Historisierung von Attributänderungen innerhalb von Dimensionstabellen über die Zeit [1, S. 95],[3, Folie 52].

**SCD Typ 1** überschreibt alte Attributwerte ohne Speicherung einer Historie [1, S. 95–97],[3, Folie 54]. Eine zeitpunktbezogene Auswertung (Time-Travel-Analyse) ist damit nicht möglich.

**SCD Typ 2** historisiert Änderungen durch das Einfügen neuer Zeilen mit definierten Gültigkeitszeiträumen [1, S. 97–100],[3, Folie 55]. Jede Änderung erzeugt einen neuen Datensatz mit eigenem Surrogate Key. Die Attribute `valid_from` und `valid_to` beschreiben den Gültigkeitszeitraum, während `is_current` die aktuell gültige Version kennzeichnet.


## 2.3 Transformation zu einem Star Schema

Die Transformation vom Staging zu einem Star Schema ist der komplexeste Teil der Datenintegration [5, Folie 11]. Abbildung 2 zeigt diese Transformation: Mehrere normalisierte Tabellen des Staging-Bereichs werden in ein denormalisiertes dimensionales Modell mit zentraler Faktentabelle und zugehörigen Dimensionstabellen überführt.


![3NF zu Star Schema](https://github.com/maximpazer/starschema-llm-demo/blob/main/images/Transformation_Star_Schema.png?raw=1)

*Abbildung 2: Transformation von 3NF zu Star Schema (Quelle: [3, Folie 21])*

Im Rahmen dieser Arbeit wird diese Modellierungsaufgabe automatisiert. Ein LLM erhält die Staging-Tabellen in Form von SQL-DDL sowie User Stories als Eingabe und generiert daraus die SQL-DDL eines Star Schemas.



---

# 3. Large Language Models für strukturierte Datengenerierung

LLMs neigen bei strukturierter Generierung zu Halluzinationen, bei denen der erzeugte Output plausibel erscheint, jedoch faktisch inkorrekt ist [6]. Für die strukturierte Codegenerierung sind solche Fehler besonders kritisch, da sie unter Umständen ungeprüft in das physische Datenmodell übernommen werden und nachgelagerte Analysen beeinträchtigen können [8].

Ursachen für Halluzinationen sind typischerweise mehrdeutige oder widersprüchliche Prompts, fehlender Kontext sowie die probabilistische Natur von LLMs, bei der statistisch wahrscheinliche Tokenfolgen gegenüber strikt regelbasierten Constraints bevorzugt werden [9].

Im Kontext von Text-to-SQL [7] wurden sogenannte **schema-based hallucinations** beobachtet, bei denen LLMs Tabellen, Spalten oder Beziehungen referenzieren, die im gegebenen Schema nicht existieren. Studien zu großen Data Warehouses (z. B. mit über 17.000 Spalten) zeigen, dass LLMs ohne explizite Schema-Informationen im Prompt häufig zu solchen Halluzinationen neigen [11]. Weitere Fehlerarten umfassen logische Inkonsistenzen (z. B. fehlerhafte JOIN-Bedingungen) sowie syntaktische Fehler (z. B. ungültige Datentypen für die Ziel-Datenbank) [10][8].

Zur Reduktion von Halluzinationen wird häufig schema-aware Prompting eingesetzt, bei dem relevante Tabellen und Spalten explizit im Prompt angegeben werden [11]. Dies erhöht die Trefferquote korrekt referenzierter Schemaelemente insbesondere bei großen Datenbanken.



---

# 4. Methodik: LLM & Metadata-Driven Engineering

Diese Arbeit verfolgt, wie in Abbildung 3 dargestellt, einen metadata-driven Ansatz zur Generierung eines Star Schemas. Das LLM erhält als Eingabe fünf User Stories sowie die DDL-Definitionen der Staging-Tabellen. Auf dieser Grundlage generiert das Modell im Zero-shot-Verfahren SQL-DDL-Statements für ein dimensionales Schema.

Die Generierung erfolgt in der SQL-Dialektvariante von DuckDB. Das resultierende Schema umfasst vier Dimensionstabellen (Fahrzeug, Händler, Kunde, Zeit) sowie eine Faktentabelle (Verkäufe). Anschließend wird das erzeugte Schema mit einem manuell erstellten Golden Standard verglichen.

![Methodik](https://github.com/maximpazer/starschema-llm-demo/blob/main/images/Methodik.png?raw=1)

*Abbildung 3: Methodischer Ablauf der Schema-Generierung*


## 4.1 Design Entscheidungen

Für die Generierung wird GPT-4o (OpenAI) über die API verwendet. Die Wahl von GPT-4o basiert auf mehreren Faktoren: GPT-4o zeigt nachgewiesene Leistungsfähigkeit bei strukturierter Code-Generierung, insbesondere bei SQL und DDL-Aufgaben [10]. Das Modell bietet zudem eine hohe Leistungsfähigkeit bei geringer Latenz. Neuere Modelle wie GPT o1 oder o3 sind primär auf multi-step reasoning optimiert, was für direkte DDL-Generierung Latenz und höhere Kosten erzeugen würde.

Der Zugriff über die API ermöglicht die Kontrolle zentraler Parameter wie der Temperatur sowie die Verwendung einer fest definierten Modellversion, was für die Reproduzierbarkeit der Experimente erforderlich ist.

In der Implementierung wird die Temperatur auf 0 gesetzt, um die Varianz der Ausgaben zu minimieren und möglichst konsistente Ergebnisse zu erhalten. Die Generierung erfolgt im Zero-shot-Verfahren ohne Few-shot-Beispiele, um die grundsätzliche Fähigkeit des Modells zur Ableitung eines Schemas aus den bereitgestellten Informationen zu untersuchen. Few-shot-Beispiele könnten die Genauigkeit erhöhen, würden jedoch gleichzeitig das Ergebnis stärker an die konkrete Promptgestaltung binden.


## 4.2 Prompt-Design

Der Prompt folgt einer strukturierten Architektur aus fünf Bausteinen:

1. **Rollendefinition**: „Du bist ein erfahrener Data Engineer mit Expertise in Kimball Dimensional Modeling“
2. **Aufgabenbeschreibung**: Erstellung eines Star Schemas mit vier Dimensionen und einer Faktentabelle; der Grain entspricht einem Verkaufsvorgang pro Zeile.
3. **Eingabedaten**: vollständige Staging-DDL sowie fünf User Stories.
4. **Explizite Constraints**: DuckDB-Syntax, Verwendung von Surrogate Keys, SCD-Typ 1/2 gemäß den User Stories sowie Denormalisierung.
5. **Ausgabeformat**: „Nur SQL-DDL, keine Erklärungen“.

Diese Struktur adressiert mehrere zentrale Herausforderungen. Die Rollendefinition zielt darauf ab, die Generierung in den Kontext des Data Warehousing einzuordnen. Die expliziten Constraints sollen die Einhaltung des gewünschten SQL-Dialekts sowie die Verwendung dimensionaler Modellierungsprinzipien unterstützen. Die Kombination aus Staging-DDL und User Stories ermöglicht es dem LLM, sowohl strukturelle Abhängigkeiten als auch semantische Anforderungen zu berücksichtigen. Der vollständige Prompt ist in Kapitel 5.3 dokumentiert.


## 4.3 Evaluation

Ein manuell erstellter Golden Standard definiert die erwarteten Modellierungsentscheidungen (SCD-Typen, Grain, Denormalisierung) auf Basis der User Stories. Die vollständige Spezifikation wird in Kapitel 5.2 beschrieben. Die Evaluation erfolgt entlang dreier Dimensionen, die aus der Aufgabenstellung sowie etablierten Evaluationsansätzen für LLM-Codegenerierung abgeleitet werden [11]:

1. **Strukturelle Korrektheit**: Bewertung, ob ein technisch ausführbares Schema erzeugt wurde (vorhandene Tabellen, syntaktisch valide DDL).
2. **Semantische Korrektheit**: Bewertung, ob die fachlichen Anforderungen korrekt in Modellierungsentscheidungen überführt wurden (SCD-Typen, Grain-Definition, Denormalisierung).
3. **Halluzinationen**: Analyse, ob nicht im Schema vorhandene Elemente referenziert oder erzeugt wurden (schema-based Halluzinationen) bzw. logisch inkonsistente Strukturen entstanden sind.

Die Dreiteilung erlaubt eine differenzierte Fehlerklassifikation: Ein Schema kann strukturell korrekt (syntaktisch valide), jedoch semantisch fehlerhaft (z. B. falsche SCD-Wahl) oder von Halluzinationen betroffen sein.


---

# 5. Praktische Implementierung

## 5.1 Staging Schema

Das Staging-Schema repräsentiert operative Quelltabellen in dritter Normalform (insgesamt sechs Tabellen: Modelle, Werke, Fahrzeuge, Händler, Kunden, Verkäufe). Die getrennte Modellierung von Modell- und Werksdaten dient dazu, die Fähigkeit des LLM zur Denormalisierung im dimensionalen Zielschema zu untersuchen.


In [3]:
STAGING_DDL = """
CREATE TABLE stg_model (
    model_id INTEGER PRIMARY KEY,
    model_name VARCHAR NOT NULL,
    brand VARCHAR NOT NULL,
    category VARCHAR NOT NULL
);

CREATE TABLE stg_plant (
    plant_id INTEGER PRIMARY KEY,
    plant_name VARCHAR NOT NULL,
    plant_city VARCHAR NOT NULL,
    plant_country VARCHAR NOT NULL
);

CREATE TABLE stg_vehicle (
    vin VARCHAR(17) PRIMARY KEY,
    model_id INTEGER NOT NULL REFERENCES stg_model(model_id),
    plant_id INTEGER NOT NULL REFERENCES stg_plant(plant_id),
    production_date DATE NOT NULL,
    base_price DECIMAL(12,2) NOT NULL
);

CREATE TABLE stg_dealer (
    dealer_id INTEGER PRIMARY KEY,
    dealer_name VARCHAR NOT NULL,
    street VARCHAR,
    city VARCHAR NOT NULL,
    region VARCHAR NOT NULL,
    state VARCHAR NOT NULL
);

CREATE TABLE stg_customer (
    customer_id INTEGER PRIMARY KEY,
    first_name VARCHAR NOT NULL,
    last_name VARCHAR NOT NULL,
    email VARCHAR,
    street VARCHAR,
    city VARCHAR,
    state VARCHAR,
    zip_code VARCHAR
);

CREATE TABLE stg_sale (
    sale_id INTEGER PRIMARY KEY,
    vin VARCHAR(17) NOT NULL REFERENCES stg_vehicle(vin),
    dealer_id INTEGER NOT NULL REFERENCES stg_dealer(dealer_id),
    customer_id INTEGER NOT NULL REFERENCES stg_customer(customer_id),
    sale_date DATE NOT NULL,
    sale_price DECIMAL(12,2) NOT NULL,
    discount_pct DECIMAL(5,2) DEFAULT 0
);
"""

# Staging-Tabellen in DuckDB anlegen und prüfen
con_staging = duckdb.connect(":memory:")
con_staging.execute(STAGING_DDL)
print("Staging-Tabellen erstellt:")
for row in con_staging.execute("SHOW TABLES").fetchall():
    cols = con_staging.execute(f"PRAGMA table_info('{row[0]}')").fetchall()
    print(f"  {row[0]:20s} ({len(cols)} Spalten)")

Staging-Tabellen erstellt:
  stg_customer         (8 Spalten)
  stg_dealer           (6 Spalten)
  stg_model            (4 Spalten)
  stg_plant            (4 Spalten)
  stg_sale             (7 Spalten)
  stg_vehicle          (5 Spalten)


## 5.2 Golden Standard

Der Golden Standard legt die erwarteten Modellierungsentscheidungen fest:

| Dimension | SCD-Typ | Begründung |
|-----------|---------|------------|
| `dim_vehicle` | Typ 2 | US-2 fordert eine Preis-Historie |
| `dim_dealer` | Typ 2 | US-3 erfordert die Nachverfolgung von Regionswechseln |
| `dim_customer` | Typ 1 | Keine historische Analyse erforderlich |
| `dim_date` | statisch | Kalenderdimension ohne fachliche Historisierung; ermöglicht Zeitanalysen nach US-1 |

*Tabelle 1: Golden Standard*

**Grain:** Eine Zeile der Faktentabelle entspricht einem einzelnen Verkaufsvorgang.  
Die normalisierten Staging-Tabellen `stg_model` und `stg_plant` werden in die Dimension `dim_vehicle` integriert.


In [4]:
GOLDEN_STANDARD_DDL = """
CREATE TABLE dim_vehicle (
    vehicle_sk INTEGER PRIMARY KEY,
    vin VARCHAR(17) NOT NULL,
    model_name VARCHAR NOT NULL,
    brand VARCHAR NOT NULL,
    category VARCHAR NOT NULL,
    plant_name VARCHAR NOT NULL,
    plant_country VARCHAR NOT NULL,
    base_price DECIMAL(12,2) NOT NULL,
    valid_from DATE NOT NULL,
    valid_to DATE,
    is_current BOOLEAN NOT NULL DEFAULT TRUE
);

CREATE TABLE dim_dealer (
    dealer_sk INTEGER PRIMARY KEY,
    dealer_id INTEGER NOT NULL,
    dealer_name VARCHAR NOT NULL,
    city VARCHAR NOT NULL,
    region VARCHAR NOT NULL,
    state VARCHAR NOT NULL,
    valid_from DATE NOT NULL,
    valid_to DATE,
    is_current BOOLEAN NOT NULL DEFAULT TRUE
);

CREATE TABLE dim_customer (
    customer_sk INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    first_name VARCHAR NOT NULL,
    last_name VARCHAR NOT NULL,
    email VARCHAR,
    city VARCHAR,
    state VARCHAR,
    zip_code VARCHAR
);

CREATE TABLE dim_date (
    date_sk INTEGER PRIMARY KEY,
    full_date DATE NOT NULL,
    day_of_month INTEGER NOT NULL,
    month INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    year INTEGER NOT NULL,
    day_of_week VARCHAR NOT NULL,
    is_weekend BOOLEAN NOT NULL
);

CREATE TABLE fact_sales (
    sale_sk INTEGER PRIMARY KEY,
    vehicle_sk INTEGER NOT NULL REFERENCES dim_vehicle(vehicle_sk),
    dealer_sk INTEGER NOT NULL REFERENCES dim_dealer(dealer_sk),
    customer_sk INTEGER NOT NULL REFERENCES dim_customer(customer_sk),
    sale_date_sk INTEGER NOT NULL REFERENCES dim_date(date_sk),
    production_date_sk INTEGER NOT NULL REFERENCES dim_date(date_sk),
    vin VARCHAR(17) NOT NULL,
    sale_price DECIMAL(12,2) NOT NULL,
    discount_pct DECIMAL(5,2),
    discount_amount DECIMAL(12,2),
    time_to_sale_days INTEGER
);
"""

# Golden Standard in DuckDB validieren
con_golden = duckdb.connect(":memory:")
con_golden.execute(GOLDEN_STANDARD_DDL)
print("Golden Standard – Tabellenstruktur:")
for row in con_golden.execute("SHOW TABLES").fetchall():
    cols = con_golden.execute(f"PRAGMA table_info('{row[0]}')").fetchall()
    col_names = [c[1] for c in cols]
    print(f"\n  {row[0]}:")
    print(f"    Spalten: {', '.join(col_names)}")

Golden Standard – Tabellenstruktur:

  dim_customer:
    Spalten: customer_sk, customer_id, first_name, last_name, email, city, state, zip_code

  dim_date:
    Spalten: date_sk, full_date, day_of_month, month, quarter, year, day_of_week, is_weekend

  dim_dealer:
    Spalten: dealer_sk, dealer_id, dealer_name, city, region, state, valid_from, valid_to, is_current

  dim_vehicle:
    Spalten: vehicle_sk, vin, model_name, brand, category, plant_name, plant_country, base_price, valid_from, valid_to, is_current

  fact_sales:
    Spalten: sale_sk, vehicle_sk, dealer_sk, customer_sk, sale_date_sk, production_date_sk, vin, sale_price, discount_pct, discount_amount, time_to_sale_days


## 5.3 Prompt-Konstruktion & LLM-Generierung

In [5]:
USER_STORIES = """
US-1: Als Sales Analyst möchte ich Verkaufszahlen pro Modell, Region und Zeitraum (Tag/Monat/Quartal) analysieren.
US-2: Als Product Manager benötige ich die Preis-Historie von Fahrzeugen, um Preisänderungen und deren Einfluss auf Verkäufe zu analysieren.
US-3: Als Regional Manager will ich Händler-Performance nach Region vergleichen, auch wenn Händler die Region wechseln.
US-4: Als Controller möchte ich durchschnittliche Rabatte pro Modell und Werk analysieren.
US-5: Als Data Scientist brauche ich die Zeit zwischen Produktion und Verkauf (Time-to-Sale).
"""

prompt = f"""Du bist ein erfahrener Data Engineer mit Expertise in Kimball Dimensional Modeling und DuckDB.

AUFGABE:
Erstelle ein Star Schema bestehend aus vier Dimensionstabellen (Fahrzeug, Händler, Kunde, Zeit) und einer Faktentabelle (Verkäufe). Das Grain der Faktentabelle ist: eine Zeile entspricht einem einzelnen Verkaufsvorgang.

STAGING SCHEMA:
{STAGING_DDL}

USER STORIES:
{USER_STORIES}

CONSTRAINTS:
- Generiere ausschließlich DuckDB-kompatible SQL DDL-Statements (CREATE TABLE).
- Verwende Surrogate Keys (INTEGER) als Primärschlüssel für alle Dimensionen und die Faktentabelle.
- Behalte Natural Keys (z.B. VIN, dealer_id) als Attribute in den Dimensionen.
- Wende SCD Typ 1 oder Typ 2 an, je nach Anforderung der User Stories. SCD Typ 2 erfordert: valid_from (DATE), valid_to (DATE), is_current (BOOLEAN).
- Denormalisiere die Dimensionen vollständig (keine Snowflake-Struktur).
- Definiere Foreign Key Beziehungen von der Faktentabelle zu allen Dimensionen.
- Die Zeitdimension soll Attribute für Tag, Monat, Quartal und Jahr enthalten.

OUTPUT:
Antworte ausschließlich mit den SQL DDL-Statements. Keine Erklärungen, kein Markdown, keine Kommentare."""

print(f"Prompt: {len(prompt)} Zeichen")
print(f"Modell: gpt-4o | Temperature: 0 | Zero-shot\n")

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_tokens=4096
)

generated_ddl = response.choices[0].message.content
print(f"Token: {response.usage.prompt_tokens} (Prompt) + {response.usage.completion_tokens} (Completion)")
print(f"\n{'='*60}")
print("GENERIERTES STAR SCHEMA:")
print('='*60)
print(generated_ddl)

Prompt: 3063 Zeichen
Modell: gpt-4o | Temperature: 0 | Zero-shot

Token: 721 (Prompt) + 363 (Completion)

GENERIERTES STAR SCHEMA:
```sql
CREATE TABLE dim_vehicle (
    vehicle_key INTEGER PRIMARY KEY,
    vin VARCHAR(17) NOT NULL,
    model_id INTEGER NOT NULL,
    model_name VARCHAR NOT NULL,
    brand VARCHAR NOT NULL,
    category VARCHAR NOT NULL,
    plant_id INTEGER NOT NULL,
    plant_name VARCHAR NOT NULL,
    plant_city VARCHAR NOT NULL,
    plant_country VARCHAR NOT NULL,
    production_date DATE NOT NULL,
    base_price DECIMAL(12,2) NOT NULL
);

CREATE TABLE dim_dealer (
    dealer_key INTEGER PRIMARY KEY,
    dealer_id INTEGER NOT NULL,
    dealer_name VARCHAR NOT NULL,
    street VARCHAR,
    city VARCHAR NOT NULL,
    region VARCHAR NOT NULL,
    state VARCHAR NOT NULL,
    valid_from DATE NOT NULL,
    valid_to DATE,
    is_current BOOLEAN NOT NULL
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    first_name VAR

## 5.4 Syntaktische Validierung

In [6]:
con_gen = duckdb.connect(":memory:")

# Markdown-Codeblöcke entfernen, falls LLM diese trotz Anweisung generiert
ddl_clean = generated_ddl.strip()
if ddl_clean.startswith("```"):
    ddl_clean = "\n".join(ddl_clean.split("\n")[1:])
if ddl_clean.endswith("```"):
    ddl_clean = ddl_clean[:-3].strip()

try:
    con_gen.execute(ddl_clean)
    tables = con_gen.execute("SHOW TABLES").fetchall()
    print(f"Syntaktische Validierung: ERFOLGREICH ({len(tables)} Tabellen)\n")

    for row in tables:
        t = row[0]
        cols = con_gen.execute(f"PRAGMA table_info('{t}')").fetchall()
        print(f"--- {t} ---")
        for c in cols:
            print(f"  {c[1]:30s} {c[2]}")
        print()
except Exception as e:
    print(f"Syntaktische Validierung: FEHLGESCHLAGEN\nFehler: {e}")

Syntaktische Validierung: ERFOLGREICH (5 Tabellen)

--- dim_customer ---
  customer_key                   INTEGER
  customer_id                    INTEGER
  first_name                     VARCHAR
  last_name                      VARCHAR
  email                          VARCHAR
  street                         VARCHAR
  city                           VARCHAR
  state                          VARCHAR
  zip_code                       VARCHAR

--- dim_dealer ---
  dealer_key                     INTEGER
  dealer_id                      INTEGER
  dealer_name                    VARCHAR
  street                         VARCHAR
  city                           VARCHAR
  region                         VARCHAR
  state                          VARCHAR
  valid_from                     DATE
  valid_to                       DATE
  is_current                     BOOLEAN

--- dim_time ---
  time_key                       INTEGER
  date                           DATE
  day                            INTE

---

# 6. Evaluation & Ergebnisse

Nachdem die syntaktische Validierung in Kapitel 5.4 gezeigt hat, dass das generierte Schema in DuckDB ausführbar ist, erfolgt nun die qualitative Bewertung. Dieses Kapitel vergleicht das LLM-generierte Schema mit dem Golden Standard anhand der in Abschnitt 4.3 definierten Kriterien.

In [7]:
def get_table_info(connection, table_name):
    """Gibt Spaltendetails einer Tabelle als Dict zurück."""
    cols = connection.execute(f"PRAGMA table_info('{table_name}')").fetchall()
    return {c[1]: c[2] for c in cols}

golden_tables = {r[0]: get_table_info(con_golden, r[0])
                 for r in con_golden.execute("SHOW TABLES").fetchall()}
gen_tables = {r[0]: get_table_info(con_gen, r[0])
              for r in con_gen.execute("SHOW TABLES").fetchall()}

print("=" * 65)
print("VERGLEICH: Golden Standard vs. LLM-generiertes Schema")
print("=" * 65)

# Tabellenabgleich
print(f"\n{'Tabelle':<20} {'Golden Standard':<22} {'LLM-generiert':<22}")
print("-" * 65)
all_tables = sorted(set(list(golden_tables.keys()) + list(gen_tables.keys())))
for t in all_tables:
    g = f"{len(golden_tables[t])} Spalten" if t in golden_tables else "—"
    l = f"{len(gen_tables[t])} Spalten" if t in gen_tables else "—"
    match = "✓" if t in golden_tables and t in gen_tables else "✗"
    print(f"  {t:<18} {g:<22} {l:<22} {match}")

# Detailvergleich pro Dimension
print(f"\n{'='*65}")
print("SPALTENVERGLEICH PRO TABELLE")
print("="*65)

table_mapping = {
    "dim_vehicle": "dim_vehicle",
    "dim_dealer": "dim_dealer",
    "dim_customer": "dim_customer",
    "dim_date": "dim_time",
    "fact_sales": "fact_sales"
}

for golden_name, gen_name in table_mapping.items():
    golden_cols = set(golden_tables.get(golden_name, {}).keys())
    gen_cols = set(gen_tables.get(gen_name, {}).keys())

    missing = golden_cols - gen_cols
    extra = gen_cols - golden_cols
    common = golden_cols & gen_cols

    print(f"\n--- {golden_name} (Golden) vs. {gen_name} (LLM) ---")
    print(f"  Gemeinsam:  {len(common):>2}  {sorted(common)}")
    if missing:
        print(f"  Fehlend:    {len(missing):>2}  {sorted(missing)}")
    if extra:
        print(f"  Zusätzlich: {len(extra):>2}  {sorted(extra)}")

VERGLEICH: Golden Standard vs. LLM-generiertes Schema

Tabelle              Golden Standard        LLM-generiert         
-----------------------------------------------------------------
  dim_customer       8 Spalten              9 Spalten              ✓
  dim_date           8 Spalten              —                      ✗
  dim_dealer         9 Spalten              10 Spalten             ✓
  dim_time           —                      6 Spalten              ✗
  dim_vehicle        11 Spalten             12 Spalten             ✓
  fact_sales         11 Spalten             8 Spalten              ✓

SPALTENVERGLEICH PRO TABELLE

--- dim_vehicle (Golden) vs. dim_vehicle (LLM) ---
  Gemeinsam:   7  ['base_price', 'brand', 'category', 'model_name', 'plant_country', 'plant_name', 'vin']
  Fehlend:     4  ['is_current', 'valid_from', 'valid_to', 'vehicle_sk']
  Zusätzlich:  5  ['model_id', 'plant_city', 'plant_id', 'production_date', 'vehicle_key']

--- dim_dealer (Golden) vs. dim_dealer (LLM) 

## 6.1 Schema-based Halluzinationen

Im Hinblick auf schema-based Halluzinationen wurden keine erfundenen Tabellen oder Spalten generiert. Sämtliche Attribute lassen sich auf das Staging-Schema zurückführen. Auch scheinbar zusätzliche Spalten im Vergleich zum Golden Standard können aus vorhandenen Informationen des Staging-Schemas abgeleitet werden.

Dieses Ergebnis steht im Einklang mit Befunden aus der Text-to-SQL-Forschung, wonach die explizite Bereitstellung von Schema-Informationen im Prompt das Auftreten solcher Halluzinationen reduziert [11].

## 6.2 Semantische Fehler

Die semantische Evaluation untersucht, ob das LLM die fachlichen Anforderungen der User Stories korrekt in Modellierungsentscheidungen überführt hat. Tabelle 2 zeigt die für SCD relevante Ausgestaltung der Dimensionstabellen auf Basis der User Stories.

| Dimension | User Story | Erwartet | Generiert | Status |
|-----------|------------|----------|-----------|--------|
| `dim_vehicle` | US-2: Preis-Historie | **SCD Typ 2** | ✗ **SCD Typ 1** (keine Versionierung) | **✗ Fehler** |
| `dim_dealer` | US-3: Regionswechsel | SCD Typ 2 | ✓ `valid_from`, `valid_to`, `is_current` | ✓ |
| `dim_customer` | keine Historie | SCD Typ 1 | ✓ keine Versionierung | ✓ |
| `dim_time` | statisch | — | ✓ keine Versionierung | ✓ |

*Tabelle 2: Vergleich der Dimensionstabellen*

Es zeigt sich, dass in `dim_vehicle` die für SCD Typ 2 notwendigen Attribute `valid_from`, `valid_to` und `is_current` fehlen. Das LLM hat die Anforderung an eine Historisierung daher nicht umgesetzt, wodurch die geforderte Preis-Historie von Fahrzeugen (US-2) nicht analysiert werden kann.

Im Gegensatz dazu wurde `dim_dealer` korrekt als SCD Typ 2 modelliert. Dies deutet darauf hin, dass implizite Anforderungen in User Stories vom LLM nicht durchgängig zuverlässig in Modellierungsentscheidungen übertragen werden.


## 6.3 Logische Fehler

Es lassen sich vier relevante Auslassungen bzw. Modellierungsprobleme identifizieren, die die Anforderungen der User Stories betreffen:

1. **Fehlender `production_date_sk`:**  
   Die Faktentabelle enthält lediglich einen Zeitschlüssel (`time_key`), der das Verkaufsdatum referenziert. US-5 fordert jedoch die Berechnung der Time-to-Sale-Kennzahl. Hierfür wäre ein zweiter Fremdschlüssel zur Zeitdimension (`production_date_sk`) erforderlich. Die hierfür notwendige Modellierungslogik wurde vom LLM nicht abgeleitet.

2. **Fehlende abgeleitete Metrik:**  
   `discount_amount` fehlt in `fact_sales`. Diese Kennzahl könnte zwar zur Laufzeit berechnet werden, jedoch empfiehlt Kimball die Vorabberechnung häufig verwendeter Metriken in der Faktentabelle.

3. **VIN als Degenerate Dimension fehlt:**  
   Das VIN-Attribut ist in der Faktentabelle nicht enthalten. Im Golden Standard ist es als Degenerate Dimension vorgesehen, da es den einzelnen Verkaufsvorgang identifiziert, ohne eine eigene Dimensionstabelle zu erfordern.

4. **`production_date` in `dim_vehicle` (Modellierungsproblem):**  
   Zeitbezogene Attribute werden im dimensionalen Modell üblicherweise über Fremdschlüssel zur Zeitdimension referenziert. Das LLM hat stattdessen das Attribut direkt aus dem Staging-Schema in die Dimension übernommen.



---

# 7. Kritische Würdigung

Der metadata-driven Ansatz zeigt mehrere zentrale Vorteile. Durch die vollständige Angabe des Staging-Schemas wurden keine schema-based hallucinations beobachtet; dies adressiert ein häufiges Problem bei LLM-basierter SQL-Generierung, insbesondere bei großen Datenbanken [11]. Zudem werden ausschließlich Metadaten (DDL) übermittelt, jedoch keine Rohdaten. Dadurch ist der Ansatz auch für Szenarien mit sensiblen Daten prinzipiell geeignet. Darüber hinaus ist der Ansatz unabhängig vom Datenvolumen, da ausschließlich Tabellendefinitionen verarbeitet werden. Positiv hervorzuheben ist die erfolgreiche SCD-Typ-2-Implementierung von `dim_dealer` (US-3): Das LLM übersetzt die Anforderung, Regionswechsel nachzuverfolgen, korrekt in eine Versionierung. Dies deutet darauf hin, dass explizit formulierte Änderungssemantiken (z. B. „wechseln“, „nachverfolgen“) vom Modell tendenziell zuverlässiger in Historisierung überführt werden.

Gleichzeitig offenbart die Evaluation Einschränkungen, die einen Produktiveinsatz ohne zusätzliche Validierung verhindern. So wurde `dim_vehicle` als SCD Typ 1 generiert, obwohl US-2 explizit eine „Preis-Historie“ fordert und damit eine Historisierung nach SCD Typ 2 erforderlich wäre. Dies zeigt eine zentrale Limitation: Das Modell kann syntaktisch korrekten Output erzeugen, der jedoch semantisch nicht mit den fachlichen Anforderungen übereinstimmt. Der Fehler legt nahe, dass implizite Modellierungsanforderungen zur Historisierung aus User Stories nicht zuverlässig abgeleitet werden. Ähnliche Inkonsistenzen bei mehrdeutigen oder variabel formulierten Anforderungen wurden auch in der Literatur beschrieben [8].

Als weitere Schwäche sind logische Auslassungen zu nennen, die die Umsetzbarkeit einzelner User Stories beeinträchtigen. Beispielsweise ist durch das Auslassen von `production_date_sk` in `fact_sales` die User Story US-5 (Time-to-Sale) nicht ohne zusätzliche Umwege bzw. Nachmodellierung realisierbar.

Der Ansatz ist damit ohne manuelle Prüfung des generierten Schemas nicht produktionsreif. In Enterprise-Kontexten mit komplexeren Anforderungen (z. B. Multi-Fact-Schemas, Bridge Tables, Ragged Hierarchies) ist zu erwarten, dass zusätzliche Fehlerklassen auftreten. Besonders kritisch sind semantische Fehler (z. B. falsche SCD-Typen), da sie bei syntaktisch korrekten und ausführbaren Schemas schwer zu erkennen sind. Ein falsch modelliertes Data Warehouse kann in der Folge zu fehlerhaften Auswertungen und Business-Entscheidungen führen (z. B. Preisanalysen auf überschriebenen statt historisierten Werten). Solche Fehler sind potenziell folgenschwerer als rein syntaktische Fehler, die typischerweise unmittelbar auffallen.

Zur Erhöhung der Zuverlässigkeit LLM-generierter Star Schemas ergeben sich folgende Richtungen:

- **Few-Shot Learning:** Beispielschemata im Prompt könnten die Erkennung von Historisierungsanforderungen (z. B. SCD Typ 2) verbessern, erhöhen jedoch den Promptumfang und reduzieren potenziell die Generalisierbarkeit der Ergebnisse.

- **Agentic Workflows:** Eine iterative Validierung durch einen regelbasierten Validator, der strukturiertes Feedback und Korrekturhinweise liefert, könnte logische und semantische Fehler reduzieren.

- **Aktuellere bzw. alternative Modelle:** Neuere oder stärker auf Reasoning ausgelegte Modelle könnten Anforderungen aus User Stories robuster in Modellierungsentscheidungen übersetzen. Ein systematischer Modellvergleich (bei gleicher Prompt- und Datengrundlage) kann zusätzliche Erkenntnisse zur Modellwahl und zu Fehlerprofilen liefern.


---

# 8. Fazit und Ausblick

Ziel dieser Arbeit war die Analyse der Generierung von Star Schemas auf Basis von User Stories und Staging-Schemas. Der Zero-shot metadata-driven Ansatz zeigt, dass LLMs im Kontext von Data Warehousing Potenzial für standardisierte Anwendungsfälle besitzen, für produktive ETL-Pipelines ohne menschliche Validierung jedoch derzeit nicht geeignet sind. Der primäre Mehrwert liegt in der Beschleunigung der initialen Schemaerstellung und nicht in einer vollständig automatisierten Modellierung.

Eine zentrale Erkenntnis ist, dass schema-aware Prompting das Auftreten erfundener Schemaelemente deutlich reduziert, während die semantische Interpretation der Anforderungen inkonsistent bleibt. LLMs verarbeiten explizit formulierte Anforderungen (z. B. „Regionswechsel nachverfolgen“) zuverlässig, erkennen jedoch nicht alle impliziten Anforderungen.

Für den praktischen Einsatz erscheinen Hybridansätze mit regelbasierter Validierung vielversprechend, bei denen ein Data Engineer die finale Prüfung des generierten Schemas übernimmt. Dies kann insbesondere bei mehreren Data Marts die Modellierungszeit signifikant reduzieren.

# 9. Quellen

[1] R. Kimball and M. Ross, "Dimensional modeling techniques," Kimball Group, 2013. [Online]. Available: http://www.kimballgroup.com/wp-content/uploads/2013/08/2013.09-Kimball-Dimensional-Modeling-Techniques11.pdf (accessed Feb. 13, 2026).

[2] M. Chen et al., "Evaluating Large Language Models Trained on Code," arXiv preprint arXiv:2107.03374, 2021. [Online]. Available: https://arxiv.org/abs/2107.03374 (accessed Feb. 14, 2026).

[3] A. Buckenhofer, "Datenmodellierung," Vorlesungsfolien, DHBW Stuttgart, WS 2025/26.

[4] R. Kimball and M. Ross, The Data Warehouse Toolkit: The Complete Guide to Dimensional Modeling, 2nd ed. New York, NY, USA: John Wiley & Sons, 2002.

[5] A. Buckenhofer, "Data Engineering," Vorlesungsfolien, DHBW Stuttgart, WS 2025/26.

[6] J. Wang and Y. Chen, "A Review on Code Generation with LLMs: Application and Evaluation," in Proc. 2023 IEEE Int. Conf. Medical Artificial Intelligence (MedAI), 2023, doi: 10.1109/MedAI59581.2023.00044.

[7] D. Gao, H. Wang, Y. Li, X. Sun, Y. Qian, B. Ding, and J. Zhou, "Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation," Proc. VLDB Endowment, vol. 17, no. 5, pp. 1132-1145, 2024, doi: 10.14778/3641204.3641221.

[8] F. Liu, Y. Liu, L. Shi, Z. Yang, L. Zhang, X. Lian, Z. Li, and Y. Ma, "Beyond Functional Correctness: Exploring Hallucinations in LLM-Generated Code," arXiv preprint arXiv:2404.00971, 2024. [Online]. Available: https://arxiv.org/abs/2404.00971 (accessed Feb. 14, 2026).

[9] D. Anh-Hoang, V. Tran, and L.-M. Nguyen, "Survey and analysis of hallucinations in large language models: attribution to prompting strategies or model behavior," Front. Artif. Intell., vol. 8, Art. no. 1622292, Sep. 2025. [Online]. Available: https://pmc.ncbi.nlm.nih.gov/articles/PMC12518350/

[10] Synscribe, "GPT-4o benchmark: detailed comparison with Claude and Gemini," Synscribe Blog, 2024. [Online]. Available: https://www.synscribe.com/blog/gpt-4o-benchmark-detailed-comparison-with-claude-and-gemini (accessed Feb. 14, 2026).

[11] M. Kothyari, D. Dhingra, S. Sarawagi, and S. Chakrabarti, "CRUSH4SQL: Collective Retrieval Using Schema Hallucination for Text2SQL," in Proc. 2023 Conf. Empirical Methods in Natural Language Processing (EMNLP), Singapore, 2023, pp. 14054–14066. [Online]. Available: https://aclanthology.org/2023.emnlp-main.868



---

# 10. Gen-AI Erlärung

Bei der Erstellung der eingereichten Arbeit habe ich auf künstlicher Intelligenz (KI) basierte Systeme benutzt:

[x] ja

[ ] nein

Falls ja: Die nachfolgend aufgeführten auf künstlicher Intelligenz (KI) basierten Systeme habe ich bei der Erstellung der eingereichten Arbeit benutzt:
1. Perplexity Pro mit Claude Sonnet 4.5
2. ChatGPT 5.2
3. GitHub Copilot (Claude Opus 4.5)

Ich erkläre, dass ich
- mich aktiv über die Leistungsfähigkeit und Beschränkungen der oben genannten KI-Systeme informiert habe,
- die aus den oben angegebenen KI-Systemen direkt oder sinngemäß übernommenen Passagen gekennzeichnet habe,
- überprüft habe, dass die mithilfe der oben genannten KI-Systeme generierten und von mir übernommenen Inhalte faktisch richtig sind,
- mir bewusst bin, dass ich als Autorin bzw. Autor dieser Arbeit die Verantwortung für die in ihr gemachten Angaben und Aussagen trage.

Die oben genannten KI-Systeme habe ich wie im Folgenden dargestellt eingesetzt:

| Arbeitsschritt in der wissenschaftlichen Arbeit | Eingesetzte(s) KI-System(e) | Beschreibung der Verwendungsweise |
|---|---|---|
| Code Generierung & Debugging | GitHub Copilot mit Claude Opus 4.5| Bezüglich der Aufgabenstellung wurden der Use Case selber entwickelt und das environment aufgesetzt. Claude Opus hat dabei unterstüzt das Staging zu implementieren zusammen mit der folgenden LLM Generierung und API-Calls. Bei Fehlern im Code hat Github Copilot Änderungsvorschläge gegeben|
| Literaturrecherche | Perplexity Pro mit Claude Sonnet 4.5 | Um wesentliche Literatur zu finden wurde Perplexity verwendet, um Paper und Artikel zu finden die für die theoretischen Inhalte notwendig waren. Dabei wurde die Deep Research Funktion verwendet. |
| Sprachliche Glättung | Perplexity Pro mit Claude Sonnet 4.5| Das Modell wurde ebenso verwendet, um sprachliche Redundanzen zu glätten, Formulierungen anzupassen und einen roten Faden mit einer runden Storyline zu entwickeln. Die vorgeschlagenen Änderungen wurden auf Korrektheit überprüft und abgeändert übernommen.  |
| Unformulierung | ChatGPT 5.2| Das Modell wurde verwendet, um Sätze sprachlich zu glätten und Redundanzen zu finden, sodass das Zeichenlimit nicht überschritten wird.  |
